# Reel Pipeline (PowerShell + Speechma)
Run top-to-bottom. This notebook builds reel video, voice, captions, and branded final output.


## Setup
Load PowerShell cell magic (one-time per session).


In [1]:
%load_ext powershell_magic

The powershell_magic extension is already loaded. To reload it, use:
  %reload_ext powershell_magic


## Step 1: Initialize Paths and Folders
Creates reel workspace folders and shared variables used by later cells.


In [10]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 1: Initialize Paths and Folders'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# ------------------------------
# 1) Define Paths + Core Variables
# ------------------------------
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"

# ------------------------------
# 2) Create Output Folders
# ------------------------------
New-Item -ItemType Directory -Force $FINAL_DIR | Out-Null
New-Item -ItemType Directory -Force $CAPTIONS_DIR | Out-Null
New-Item -ItemType Directory -Force $VOICE_DIR | Out-Null
Write-Host "Paths initialized"

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


Paths initialized


## Step 2: Enter Speechma Narration Text
Paste text in the output textbox and click **Save Text**.


In [11]:
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets
from datetime import datetime
import subprocess
import json

PROJECT_ROOT = Path(r"C:\Users\saura\Documents\youtubeVideoAgent")
TOPIC = "corey-wayne"
now = datetime.now()
DATE = now.strftime("%Y-%m-%d")
HOUR = now.strftime("%H")
WORKSPACE = PROJECT_ROOT / "assets" / "reels" / f"{DATE}_{HOUR}_{TOPIC}"

SCRIPT_DIR = WORKSPACE / "script"
VOICE_DIR = WORKSPACE / "voice"
META_DIR = WORKSPACE / "meta"
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
VOICE_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

SCRIPT_FROM_CELL = SCRIPT_DIR / "script_from_cell.txt"
INPUT_FROM_CELL = VOICE_DIR / "speechma_input_from_cell.txt"
SETTINGS_JSON = META_DIR / "speechma_settings.json"

saved = {'voiceLabel': '', 'pitch': 0, 'speed': 25, 'volume': 200}
if SETTINGS_JSON.exists():
    try:
        old = json.loads(SETTINGS_JSON.read_text(encoding='utf-8'))
        for k in saved:
            if k in old:
                saved[k] = old[k]
    except Exception:
        pass

def detect_speechma_page_id():
    try:
        raw = subprocess.check_output(['browseros-cli', 'pages', '--json'], text=True)
        obj = json.loads(raw)
        for p in obj.get('pages', []):
            if 'speechmapro.com' in str(p.get('url', '')):
                return int(p.get('pageId'))
    except Exception:
        pass
    return 0

def load_voices_auto():
    pid = detect_speechma_page_id()
    if pid <= 0:
        return []
    js = r'''(() => {
      const out = [];
      const seen = new Set();
      const nodes = Array.from(document.querySelectorAll('[data-voice-name], .voice-item, .voice-card, .voice-option, li, button, a, div'));
      for (const n of nodes) {
        const txt = (n.textContent || '').replace(/\s+/g, ' ').trim();
        if (!txt || txt.length < 3 || txt.length > 120) continue;
        const low = txt.toLowerCase();
        if (!(low.includes('english') || low.includes('hindi') || low.includes('spanish') || low.includes('multilingual') || low.includes('male') || low.includes('female') || low.includes('india') || low.includes('us') || low.includes('uk'))) continue;
        if (seen.has(txt)) continue;
        seen.add(txt);
        out.push(txt);
      }
      return { voices: out.slice(0, 400) };
    })()'''
    try:
        raw = subprocess.check_output(['browseros-cli', 'eval', '--page', str(pid), js], text=True)
        s = raw.strip()
        a, b = s.find('{'), s.rfind('}')
        if a < 0 or b <= a:
            return []
        obj = json.loads(s[a:b+1])
        return [v for v in obj.get('voices', []) if isinstance(v, str)]
    except Exception:
        return []

title = widgets.HTML(f'<b>Step 2: Narration + Voice Settings</b><br><code>{WORKSPACE}</code>')
text_box = widgets.Textarea(value='', placeholder='Paste your Speechma narration text here...', layout=widgets.Layout(width='100%', height='220px'))

load_voices_btn = widgets.Button(description='Load Voices', button_style='info', icon='refresh')
load_default_btn = widgets.Button(description='Load Default Setting', button_style='warning', icon='download')

saved_voice = str(saved.get('voiceLabel') or '')
voice_opts = [('Default (Current Speechma voice)', '')]
if saved_voice:
    voice_opts.append((f'Saved: {saved_voice}', saved_voice))
voice_dropdown = widgets.Dropdown(options=voice_opts, value=(saved_voice if saved_voice else ''), description='Voice:', layout=widgets.Layout(width='95%'))

pitch_slider = widgets.IntSlider(value=int(saved.get('pitch', 0)), min=-100, max=100, step=1, description='Pitch:')
speed_slider = widgets.IntSlider(value=int(saved.get('speed', 25)), min=-100, max=100, step=1, description='Speed:')
volume_slider = widgets.IntSlider(value=int(saved.get('volume', 200)), min=0, max=200, step=1, description='Volume:')

save_btn = widgets.Button(description='Save Text + Settings', button_style='success', icon='save')
status = widgets.Output()

def on_load_voices(_):
    with status:
        status.clear_output()
        voices = load_voices_auto()
        if not voices:
            print('No voices detected. Open Speechma voice panel and try again.')
            return
        opts = [('Default (Current Speechma voice)', '')] + [(v, v) for v in voices]
        if saved_voice and saved_voice not in [v for _, v in opts]:
            opts.append((f'Saved: {saved_voice}', saved_voice))
        voice_dropdown.options = opts
        if saved_voice:
            voice_dropdown.value = saved_voice
        print(f'Loaded {len(voices)} voice options from active Speechma tab.')

def on_load_default(_):
    with status:
        status.clear_output()
        if not SETTINGS_JSON.exists():
            print('No saved defaults found yet.')
            return
        cfg = json.loads(SETTINGS_JSON.read_text(encoding='utf-8'))
        pitch_slider.value = int(cfg.get('pitch', 0) or 0)
        speed_slider.value = int(cfg.get('speed', 25) or 25)
        volume_slider.value = int(cfg.get('volume', 200) or 200)
        v = str(cfg.get('voiceLabel', '') or '')
        if v and v not in [x for _, x in voice_dropdown.options]:
            voice_dropdown.options = list(voice_dropdown.options) + [(f'Saved: {v}', v)]
        voice_dropdown.value = v if v else ''
        print('Loaded default settings.')

def on_save(_):
    with status:
        status.clear_output()
        text = text_box.value.replace('\r', '').strip()

        if not text:
            SCRIPT_FROM_CELL.write_text('', encoding='utf-8')
            INPUT_FROM_CELL.write_text('', encoding='utf-8')
            print('Narration text is empty. Cleared saved speech text. Step 3 will be blocked until you save non-empty text.')
        else:
            SCRIPT_FROM_CELL.write_text(text + '\n', encoding='utf-8')
            INPUT_FROM_CELL.write_text(text + '\n', encoding='utf-8')

        settings = {
            'pageId': 0,
            'voiceLabel': str(voice_dropdown.value or ''),
            'pitch': int(pitch_slider.value),
            'speed': int(speed_slider.value),
            'volume': int(volume_slider.value),
        }
        SETTINGS_JSON.write_text(json.dumps(settings, indent=2), encoding='utf-8')
        print('Saved as default settings:', settings)

load_voices_btn.on_click(on_load_voices)
load_default_btn.on_click(on_load_default)
save_btn.on_click(on_save)

display(widgets.VBox([
    title,
    text_box,
    widgets.HBox([load_voices_btn, load_default_btn]),
    voice_dropdown,
    pitch_slider,
    speed_slider,
    volume_slider,
    save_btn,
    status,
]))


## Step 3: Generate Voice with Speechma API
Uses saved text from Step 2 and produces `voice/voice_v1.mp3` in the reel workspace.


In [4]:
from pathlib import Path
from datetime import datetime
import requests
import shutil
import time
import subprocess
import json
from IPython.display import display
import ipywidgets as widgets

PROJECT_ROOT = Path(r"C:\Users\saura\Documents\youtubeVideoAgent")
TOPIC = "corey-wayne"
API_URL = "http://127.0.0.1:8787"
now = datetime.now()
DATE = now.strftime("%Y-%m-%d")
HOUR = now.strftime("%H")
WORKSPACE = PROJECT_ROOT / "assets" / "reels" / f"{DATE}_{HOUR}_{TOPIC}"
SCRIPT_FROM_CELL = WORKSPACE / "script" / "script_from_cell.txt"
VOICE_DIR = WORKSPACE / "voice"
VOICE_TARGET = VOICE_DIR / "voice_v1.mp3"
SETTINGS_JSON = WORKSPACE / "meta" / "speechma_settings.json"

bar = widgets.IntProgress(value=0, min=0, max=100, description='Step 3:', bar_style='info', layout=widgets.Layout(width='70%'))
status = widgets.HTML('Preparing Speechma API...')
display(widgets.VBox([bar, status]))

def copy_with_retry(src: Path, dst: Path, retries=12, delay=1.0):
    src = src.resolve()
    dst = dst.resolve()
    if src == dst:
        return 'same'
    last_err = None
    for _ in range(retries):
        try:
            shutil.copy2(str(src), str(dst))
            return 'copied'
        except PermissionError as e:
            last_err = e
            time.sleep(delay)
    raise last_err if last_err else RuntimeError('copy failed')

def api_alive(url: str) -> bool:
    try:
        r = requests.get(url + '/health', timeout=2)
        return r.ok
    except Exception:
        return False

def try_start_server():
    creationflags = 0x08000000
    candidates = [
        ['npm.cmd', 'run', 'speechma:api'],
        ['powershell', '-NoProfile', '-Command', 'npm run speechma:api'],
        ['node', str(PROJECT_ROOT / 'scripts' / 'speechma_api_server.mjs')],
    ]
    last_error = None
    for cmd in candidates:
        try:
            return subprocess.Popen(cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, creationflags=creationflags, shell=False)
        except FileNotFoundError as e:
            last_error = e
    raise RuntimeError(f'Could not start Speechma API. Last error: {last_error}')

def ensure_api_running():
    if api_alive(API_URL):
        return None
    proc = try_start_server()
    for _ in range(40):
        if api_alive(API_URL):
            return proc
        time.sleep(1)
    raise RuntimeError('Speechma API did not start on 127.0.0.1:8787.')

bar.value = 10
status.value = 'Checking local Speechma API...'
ensure_api_running()

if not SCRIPT_FROM_CELL.exists():
    bar.bar_style = 'danger'
    status.value = f"Missing input file: <code>{SCRIPT_FROM_CELL}</code>. Run Step 2 first."
    raise FileNotFoundError(str(SCRIPT_FROM_CELL))

text_for_speechma = SCRIPT_FROM_CELL.read_text(encoding='utf-8').strip()
if not text_for_speechma:
    bar.bar_style = 'danger'
    status.value = 'Step 3 blocked: narration text is blank. Paste text in Step 2 and save again.'
    raise RuntimeError('Narration text is blank in script_from_cell.txt')

settings = {'pageId': 0, 'voiceLabel': '', 'pitch': 0, 'speed': 25, 'volume': 200}
if SETTINGS_JSON.exists():
    loaded = json.loads(SETTINGS_JSON.read_text(encoding='utf-8'))
    settings.update({k: loaded.get(k, settings[k]) for k in settings.keys()})

bar.value = 25
status.value = 'Preparing payload...'
payload = {
    'workspacePath': str(WORKSPACE),
    'scriptPath': str(SCRIPT_FROM_CELL),
    'pageId': int(settings.get('pageId') or 0),
    'voiceLabel': str(settings.get('voiceLabel') or ''),
    'pitch': int(settings.get('pitch') or 0),
    'speed': int(settings.get('speed') or 25),
    'volume': int(settings.get('volume') or 200),
}

bar.value = 45
status.value = 'Calling local Speechma API...'
resp = requests.post(API_URL + '/speechma/run', json=payload, timeout=600)
data = resp.json()
if not data.get('ok'):
    bar.bar_style = 'danger'
    status.value = f"Speechma API failed: <code>{data.get('error','unknown error')}</code>"
    raise RuntimeError(data.get('error', 'Speechma API failed'))

bar.value = 80
status.value = 'Finalizing voice file...'
VOICE_DIR.mkdir(parents=True, exist_ok=True)
api_out = Path(data.get('outputVoicePath', '')) if data.get('outputVoicePath') else None
if api_out and api_out.exists():
    result = copy_with_retry(api_out, VOICE_TARGET)
else:
    downloads = Path.home() / 'Downloads'
    candidates = sorted(downloads.glob('speechma_audio_*.mp3'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        bar.bar_style = 'danger'
        status.value = 'No Speechma MP3 found in API output or Downloads.'
        raise RuntimeError('No Speechma MP3 found')
    result = copy_with_retry(candidates[0], VOICE_TARGET)

bar.value = 100
bar.bar_style = 'success'
status.value = f"Done ({result}). Voice ready at <code>{VOICE_TARGET}</code>"
data


{'ok': True,
 'pageId': 28,
 'scriptPath': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-14_23_corey-wayne\\script\\script_from_cell.txt',
 'inputPath': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-14_23_corey-wayne\\voice\\speechma_input_v1.txt',
 'outputVoicePath': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-14_23_corey-wayne\\voice\\voice_v1.mp3',
 'proofDir': 'C:\\Users\\saura\\Documents\\youtubeVideoAgent\\assets\\reels\\2026-05-14_23_corey-wayne\\analysis\\speechma_proof_api',
 'downloadedFile': 'C:\\Users\\saura\\Downloads\\speechma_audio_Brian Multilingual_at_11_48_17 PM_on_May_14th_2026.mp3',
 'durationSeconds': 48.408,
 'settings': {'pitch': 0, 'speed': 5, 'volume': 200, 'voiceLabel': ''},
 'matchPhraseUsed': 'Done. I restored your custom working variant (not plain GitHub) for Speechma:',
 'topic': None,
 'sourceRoot': None,
 'evidencePath': None}

## Step 3.5: Calculate Scene Count from Voice Length
Uses `voice/voice_v1.mp3` duration and computes rounded scene counts for 6s and 10s pacing.


In [12]:
from pathlib import Path
from datetime import datetime
import subprocess

PROJECT_ROOT = Path(r"C:\Users\saura\Documents\youtubeVideoAgent")
TOPIC = "corey-wayne"
now = datetime.now()
DATE = now.strftime("%Y-%m-%d")
HOUR = now.strftime("%H")
WORKSPACE = PROJECT_ROOT / "assets" / "reels" / f"{DATE}_{HOUR}_{TOPIC}"
VOICE_MP3 = WORKSPACE / "voice" / "voice_v1.mp3"

if not VOICE_MP3.exists():
    raise FileNotFoundError(f"Missing voice file: {VOICE_MP3}. Run Step 3 first.")

FFPROBE = PROJECT_ROOT / "tools" / "ffmpeg" / "ffmpeg-8.1.1-essentials_build" / "bin" / "ffprobe.exe"
raw = subprocess.check_output([
    str(FFPROBE), "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(VOICE_MP3)
], text=True).strip()
duration_sec = float(raw)

scenes_6s = round(duration_sec / 6)
scenes_10s = round(duration_sec / 10)

print(f"Voice duration: {duration_sec:.2f} sec")
print(f"6s scenes needed: {scenes_6s}")
print(f"10s scenes needed: {scenes_10s}")

{"voice_duration_sec": duration_sec, "scene_count_6s": scenes_6s, "scene_count_10s": scenes_10s}


Voice duration: 48.41 sec
6s scenes needed: 8
10s scenes needed: 5


{'voice_duration_sec': 48.408, 'scene_count_6s': 8, 'scene_count_10s': 5}

## Step 4: Build Base Video and Mux Voice
Creates concat file, stitches scenes, then combines stitched video with generated voice.


In [2]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 4: Build Base Video and Mux Voice'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Build concat file in scene order
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
Get-ChildItem $SOURCE -Filter "*.mp4" |
Sort-Object { [int](($_.BaseName -split '_')[0]) } |
ForEach-Object { "file '$($_.FullName.Replace('\\','/'))'" } | Set-Content $CONCAT
Write-Host "Concat file created:" $CONCAT

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


Concat file created: C:\Users\saura\Downloads\grok-folder-1\concat.txt


In [14]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 4: Build Base Video and Mux Voice'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Stitch all scene clips into one base video
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
& $FFMPEG -f concat -safe 0 -i $CONCAT -c copy $STITCHED_VIDEO

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


ffmpeg version 8.1.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 15.2.0 (Rev13, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopen

In [16]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 4: Build Base Video and Mux Voice'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Move latest downloaded MP3 into reel voice path, then mux with video
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
Move-Item (Get-ChildItem "$env:USERPROFILE\Downloads\*.mp3" | Sort-Object LastWriteTime -Descending | Select-Object -First 1).FullName $AUDIO -Force
& $FFMPEG -i $STITCHED_VIDEO -i $AUDIO -c:v copy -c:a aac -map 0:v:0 -map 1:a:0 -shortest $VOICED_VIDEO

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


Move-Item : Cannot bind argument to parameter 'Path' because it is null.
At line:26 char:11
+ Move-Item (Get-ChildItem "$env:USERPROFILE\Downloads\*.mp3" | Sort-Ob ...
+           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    + CategoryInfo          : InvalidData: (:) [Move-Item], ParameterBindingValidationException
    + FullyQualifiedErrorId : ParameterArgumentValidationErrorNullNotAllowed,Microsoft.PowerShell.Commands.MoveItemCom 
   mand
 
ffmpeg version 8.1.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 15.2.0 (Rev13, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --ena

## Step 5: Generate Captions
Creates word timestamps, SRT, and animated ASS captions.


In [3]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 5: Generate Captions'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Generate word-level timestamps from audio
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
cd $PROJECT_ROOT
python scripts\extract_word_timestamps.py --audio $AUDIO --out $WORDS

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-14_23_corey-wayne\captions\word_timestamps.json


In [4]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 5: Generate Captions'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Build SRT from word timestamps
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
cd $PROJECT_ROOT
python scripts\build_srt_from_words.py --words $WORDS --out $SRT

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-14_23_corey-wayne\captions\captions.srt


In [5]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 5: Generate Captions'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Build animated ASS captions from SRT + word timestamps
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
cd $PROJECT_ROOT
python scripts\build_wordtimed_ass.py --srt $SRT --words $WORDS --out $ASS --preset logicloom_ref

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-14_23_corey-wayne\captions\captions.ass


## Step 6: Burn Captions and Add Branding
Burns ASS captions onto video, adds watermark, and prints final output path.


In [6]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 6: Burn Captions and Add Branding'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Burn ASS captions into the voiced video
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
$ASS_FILTER=$ASS.Replace("\","/").Replace(":","\:")
& $FFMPEG -i $VOICED_VIDEO -vf "ass='$ASS_FILTER'" -c:a copy $FINAL_OUTPUT

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


ffmpeg version 8.1.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 15.2.0 (Rev13, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopen

In [12]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 6: Burn Captions and Add Branding'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Apply branding/watermark
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
powershell -ExecutionPolicy Bypass -File "$PROJECT_ROOT\scripts\add_logicloom_watermark.ps1" `
 -InputVideo "$FINAL_OUTPUT" `
 -OutputVideo "$BRANDED_OUTPUT" `
 -LogoPath "$PROJECT_ROOT\assets\branding\playbook_logo_nobg.png"

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


ffmpeg version 8.1.1-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 15.2.0 (Rev13, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-openal --enable-libgme --enable-libopenmpt --enable-libopen

In [13]:
%%powershell
$ProgressPreference = 'Continue'
$__activity = 'Step 6: Burn Captions and Add Branding'
Write-Progress -Activity $__activity -Status 'Starting...' -PercentComplete 10

# Final output path
$PROJECT_ROOT="C:\Users\saura\Documents\youtubeVideoAgent"
$DATE=Get-Date -Format "yyyy-MM-dd"
$HOUR=Get-Date -Format "HH"
$TOPIC="corey-wayne"

$REEL_ROOT="$PROJECT_ROOT\assets\reels\$DATE`_$HOUR`_$TOPIC"
$FINAL_DIR="$REEL_ROOT\final"
$CAPTIONS_DIR="$REEL_ROOT\captions"
$VOICE_DIR="$REEL_ROOT\voice"
$SOURCE="C:\Users\saura\Downloads\grok-folder-1"
$AUDIO="$VOICE_DIR\voice_v1.mp3"
$FFMPEG="$PROJECT_ROOT\tools\ffmpeg\ffmpeg-8.1.1-essentials_build\bin\ffmpeg.exe"
$CONCAT="$SOURCE\concat.txt"
$STITCHED_VIDEO="$FINAL_DIR\scenes_stitched.mp4"
$VOICED_VIDEO="$FINAL_DIR\scenes_stitched_voiced.mp4"
$WORDS="$CAPTIONS_DIR\word_timestamps.json"
$SRT="$CAPTIONS_DIR\captions.srt"
$ASS="$CAPTIONS_DIR\captions.ass"
$FINAL_OUTPUT="$FINAL_DIR\final_captioned.mp4"
$BRANDED_OUTPUT="$FINAL_DIR\final_captioned_branded.mp4"
Write-Host "FINAL VIDEO CREATED SUCCESSFULLY"
Write-Host $BRANDED_OUTPUT

Write-Progress -Activity $__activity -Status 'Finalizing...' -PercentComplete 95
Write-Progress -Activity $__activity -Completed


FINAL VIDEO CREATED SUCCESSFULLY
C:\Users\saura\Documents\youtubeVideoAgent\assets\reels\2026-05-15_00_corey-wayne\final\final_captioned_branded.mp4
